[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/magnetic_resonance/magnetic_resonance_space.ipynb)

# Magnetic Resonance

This example computes magnetic resonance: spins in a magnetic field precess about it, electron spins in electron spin resonance, nuclei in NMR and MRI, and the spins used as qubits. A radio-frequency field near the precession frequency tips them, and left to themselves they relax, back toward the field over a time T1 and out of step with each other over a time T2. What an experimenter measures follows: how a driven spin moves, the absorption line and how strong driving broadens it, and the spin echo that separates true dephasing from an uneven field.

A spin's state is modelled as half of one plus its Bloch vector, a scalar plus a vector of the algebra of space, with the pseudoscalar `I = mv.xyz` as the unit by which the fields turn it; that pseudoscalar fixes a handedness the spins do not have, and under a mirror this formulation runs the precession backwards. The generator, turning and relaxation together, is one map from states to their rates of change, and the evolution over a short step is a map built from it once. A pulse is a rotor acting by its sandwich, so a whole echo experiment is a composition of maps, built before any state is put in, and the sample's map is the average of its spins' maps.

In matrix notation the state reads as a 2×2 Hermitian density matrix, the reverse as the Hermitian conjugate, $I$ as the imaginary unit $i$, $x$ times the state against the field as the raising operator $\sigma_+$, and the generator as the Lindblad superoperator on the flattened density matrix, $\dot\rho = -i[H, \rho] + \sum_k \big(L_k \rho L_k^\dagger - \tfrac12 \{L_k^\dagger L_k, \rho\}\big)$; restricted to the Bloch vector it reads as the Bloch equations, $\dot{\mathbf r} = \boldsymbol\omega \times \mathbf r - \big(r_x / T_2,\ r_y / T_2,\ (r_z - 1) / T_1\big)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Generator, Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_figure, show_animation
from examples.quantum.magnetic_resonance import render

np.set_printoptions(precision=3, suppress=True)

# The geometric algebra of three-dimensional space.
ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
# A spin's state is its own reverse: a scalar plus a vector, (1 + Bloch vector) / 2.
State = ga.gatype.self_reverse()                       # 1 x y z
Rates = ga.gatype((State, State))                      # State <- State
# What becomes of each state over a stretch of time.
Evolution = ga.gatype((State, State))                  # State <- State
# A relaxation process is a vector plus a bivector, v + I w.
Process = ga.gatype.from_grades([1, 2])                # x y z yz zx xy
Rotor = ga.gatype.rotor()
# The pseudoscalar squares to minus one and commutes with everything.
I = mv.xyz                                             # [] Pseudoscalar
one = mv.scalar([1.0])                                 # [] Scalar
# The relaxation times in µs, typical of an electron spin in a solid; frequencies are in rad/µs.
T1, T2 = 10.0, 4.0

## 1. The state and how it changes

A spin's state, or that of many identical spins, is half of one plus a vector, the Bloch vector: of length one for a single spin, pointing along it, and shorter for a mixture. Two things change it. The fields turn it, in the frame that turns with the drive, about the vector of the detuning along z and the drive along x. Relaxation acts through processes, each moving part of the state elsewhere by its sandwich: x times the state of a spin against the field turns that part over onto the field, and the half turn in the plane xy scrambles the phase, the same process as the vector z up to a phase that cancels from the sandwich. Over `State`, the space of all states, the turning and the relaxing together are one linear map, the generator.

In [ ]:
# A detuning and a drive, in rad/µs, and a Bloch vector, to see the generator act on one state.
detuning, drive = 0.7, 1.3
bloch = mv.vector([0.4, -0.2, 0.6])

In [ ]:
def state(bloch: Vector) -> State:
    """The state with the given Bloch vector."""
    return 0.5 * (one + bloch)                                                  # [...] State


def relaxation(process: Process) -> Rates:
    """The change of a state due to a relaxation process L, as a map from states to their rates of change."""
    back = process.reverse().symmetric_reverse_product()                        # [] State
    return (process >> State) - back.anticommutator(State)                      # State <- State


# x times the state of a spin against the field: it turns that part of the state over onto the field.
raise_toward_field = mv.x * state(-mv.z)                                        # [] Process


def generator(detuning: np.ndarray, drive: np.ndarray) -> Rates:
    """The rate of change of a state, in the frame that turns with the drive."""
    hamiltonian = 0.5 * (mv.z * detuning + mv.x * drive)                        # [...] Vector
    turning = -2 * I * hamiltonian.commutator(State)                            # [...] Vector <- State
    relaxing = (1 / T1) * relaxation(raise_toward_field) + 0.5 * (1 / T2 - 1 / (2 * T1)) * relaxation(mv.xy)   # State <- State
    return turning + relaxing                                                   # [...] State <- State


rates = generator(detuning, drive)                                             # [] State <- State
change = rates(state(bloch))                                                    # [] State

In [ ]:
# The change of the Bloch vector:
print("generator:", render.components(change))

## 2. The steady state

Under a steady drive a spin turns about the drive while relaxation pulls it back, and it spirals into a state that no longer changes, where the generator vanishes. The generator never changes a state's scalar part, so it is singular; with that part pinned, the steady state is a single solve. What becomes of every state over a short time is the exponential of the generator, a map built once and applied at every step. The picture follows three spins at different detunings from the field axis into the states the solve predicts.

In matrix notation the evolution over $dt$ reads as the superoperator exponential $e^{\mathcal{L}\,dt}$; truncated at fourth order it is the classical fourth-order Runge-Kutta step of a linear equation.

In [ ]:
# The detunings followed, the drive, the step, and the steps: 40 µs in all.
detunings = np.array([0.0, 0.5, 1.0])
nutation_drive = 1.0
nutation_dt = 0.02
n_nutation_steps = 2000

In [ ]:
def steady(rates: Rates) -> State:
    """The state that no longer changes, with its scalar part pinned to one half."""
    scalar_part = one.scalar_product(State)                                    # Scalar <- State
    return (rates + one * scalar_part).solve(one * 0.5)                         # [...] State


def evolution(rates: Rates, dt: float) -> Evolution:
    """What becomes of each state over a time dt: exp(dt L), to fourth order in dt."""
    small = rates * dt                                                          # [...] State <- State
    term = total = State
    for order in range(1, 5):
        # The next term of the exponential series: the last one composed with small, over the order.
        term = small(term) / order                                              # [...] State <- State
        total = total + term
    return total                                                                # [...] State <- State


def evolve(each_step: Evolution, rho: State, steps: int) -> Generator[State, None, State]:
    """The state before each of the given number of steps; returns the state after the last."""
    for _ in range(steps):
        yield rho                                                               # [...] State
        rho = each_step(rho)
    return rho


driven = generator(detunings, nutation_drive)                                  # [detunings] State <- State
# All spins start along the field.
along_field = state(mv.z).broadcast_to(detunings.shape)                         # [detunings] State
settled = steady(driven)                                                        # [detunings] State
save_figure(render.draw_nutation(evolve(evolution(driven, nutation_dt), along_field, n_nutation_steps), settled), "resonance_nutation");

## 3. Line shapes

A magnetic resonance spectrum records how much energy the spins absorb from the drive at each detuning: the part of the steady Bloch vector a quarter turn behind the drive, along minus y. The part in step with it, along x, is the dispersion. Because the steady state is a single solve, the whole spectrum for several drive strengths is one batched call. With a weak drive the absorption line is as narrow as the dephasing allows; driven harder, the spins cannot keep up, and the line saturates and broadens.

In the component form of the Bloch equations the steady state reads $r_z = (1 + \Delta^2 T_2^2)/d$ and $-r_y = \Omega T_2 / d$, with $d = 1 + \Delta^2 T_2^2 + \Omega^2 T_1 T_2$, and the half-width as $\sqrt{1 + \Omega^2 T_1 T_2}\,/\,T_2$.

In [ ]:
# The detunings swept, in rad/µs, and the drive strengths.
sweep = np.linspace(-3.0, 3.0, 241)
drives = np.array([0.05, 0.3, 1.0])

In [ ]:
spectrum = steady(generator(sweep[:, None], drives[None, :]))   # [sweep, drives] State
save_figure(render.draw_lines(sweep, drives, spectrum), "resonance_lines");

## 4. Spin echo

In a real sample the field is not quite uniform, so each spin has its own detuning. Tipped into the transverse plane by a short pulse, the spins fan out, and their mean, the signal a pick-up coil sees, vanishes long before `T2`. A second pulse that turns every spin half a turn about x reverses the fan: each spin retraces its drift, and at twice the delay they line up again, having lost only the true dephasing. A short pulse is a rotor, acting on a state by its sandwich: the echo tips the spins, waits, turns them over and waits again.

In the notation of NMR the Hahn echo amplitude at twice the delay $\tau$ reads as $e^{-2\tau/T_2}$.

In [ ]:
# The spins' detunings: their spread in rad/µs, their number, and the seed; the delay before the second pulse
# and the step, in µs; the steps followed, 8 µs in all; the steps per frame, and the duration of a frame.
spread, n_spins, seed = 2.0, 300, 0
delay, dt = 2.5, 0.01
n_echo_steps = 800
n_steps_per_frame = 8
duration_ms = 40

In [ ]:
def pulse(angle: float) -> Rotor:
    """A short strong pulse along x, turning the Bloch vector about x by the angle."""
    return (I * mv.x * (-angle / 2)).exp()                                      # [] Rotor


def echo(each_step: Evolution, rho: State, before: int, after: int) -> Iterator[State]:
    """The states through a spin echo: tipped onto -y, left for the given number of steps, turned
    half a turn about x, and left again."""
    rho = yield from evolve(each_step, pulse(np.pi / 2) >> rho, before)
    yield from evolve(each_step, pulse(np.pi) >> rho, after)


scattered = np.random.default_rng(seed).normal(0.0, spread, n_spins)            # [spins] rad/µs
# No drive between the pulses; the spins start along the field.
free = generator(scattered, 0.0)                                                # [spins] State <- State
along_field = state(mv.z).broadcast_to(scattered.shape)                         # [spins] State
waits = round(delay / dt)
# The spins seen from above the field, one frame every few steps, beside the signal so far.
frames = render.animate_echo(scattered, echo(evolution(free, dt), along_field, waits, n_echo_steps - waits), dt, delay, T2, n_steps_per_frame)
show_animation(frames, "resonance_echo", duration_ms)

## 5. The experiment as one map

The evolution over one step is a map from state to state. Composed with itself it spans two steps, then four, and so on. The pulses are maps too, their sandwiches over `State`, the space of all states, so a pulse sequence is a composition of maps, built before any state is put in. Each spin has its own map, and the sample's is their average: one map from spin state to spin state for the whole experiment. Built for delays from 10 ns to 10 µs at once, the free decay is gone within a microsecond, while the echo fades only through the true dephasing.

In matrix notation the sample's echo map reads as an average over the detunings of a product of 4×4 superoperators, $\big\langle e^{\mathcal{L}\tau}\, \mathcal{R}_\pi\, e^{\mathcal{L}\tau}\, \mathcal{R}_{\pi/2} \big\rangle$, whose echo decays as $e^{-t/T_2}$.

In [ ]:
# The doublings of the step: delays from one step to a thousand and twenty-four.
n_doublings = 11

In [ ]:
def doublings(span: Evolution, count: int) -> Iterator[Evolution]:
    """The evolution over its own span, then over twice and four times that span, and so on."""
    for _ in range(count):
        yield span
        span = span(span)


# The evolution over one step, doubled for delays from dt to 1024 dt.
waiting = stack(list(doublings(evolution(free, dt), n_doublings)))             # [delays, spins] State <- State
# The times after the first pulse.
times = 2 * dt * 2.0 ** np.arange(n_doublings)                                  # µs
# The pulses as maps over `State`, the space of all states.
tip = pulse(np.pi / 2) >> State                                                  # [] State <- State
turn = pulse(np.pi) >> State                                                     # [] State <- State
# Tip, wait, turn, wait: the echo. Tip and wait twice as long: the free decay.
echo = waiting(turn(waiting(tip)))                                               # [delays, spins] State <- State
decay = waiting(waiting(tip))                                                    # [delays, spins] State <- State
# Every spin has its own map; the sample's is their average.
sample_echo = echo.mean(axis=-1)                                                 # [delays] State <- State
sample_decay = decay.mean(axis=-1)                                               # [delays] State <- State
echoed = sample_echo(state(mv.z))                                                # [delays] State
faded = sample_decay(state(mv.z))                                                # [delays] State
save_figure(render.draw_decays(times, echoed, faded, T2), "resonance_echo_decay");

In [ ]:
# checks
# The generator is the Bloch equations; the steady states do not change and are the textbook ones;
# the echo built as a map has lost only the true dephasing.
r = render.components(state(bloch))
bloch_equations = np.cross([drive, 0.0, detuning], r) - np.array([r[0] / T2, r[1] / T2, (r[2] - 1) / T1])
np.testing.assert_allclose(render.components(change), bloch_equations, atol=1e-12)
np.testing.assert_allclose(driven(settled).kernel, 0.0, atol=1e-12)
D, W = sweep[:, None], drives[None, :]
d = 1 + (D * T2) ** 2 + W**2 * T1 * T2
components = render.components(spectrum)
np.testing.assert_allclose(components[..., 2], (1 + (D * T2) ** 2) / d, atol=1e-12)
np.testing.assert_allclose(-components[..., 1], W * T2 / d, atol=1e-12)
np.testing.assert_allclose(render.transverse(echoed), np.exp(-times / T2), rtol=1e-5)